# Recálculo de iWUE con Γ\* dependiente de temperatura (Bernacchi et al. 2001)

**Entrada:** `data_for_R.xlsx` · **Salida:** `data_for_R_Bernacchi.xlsx`

Este cuaderno hace tres cosas:

1. **Limpia** espacios en blanco sobrantes en las columnas de texto (`Weinmania `, `Hedyosmun `, `Clarisia `, `CaNu `, y el nombre de columna `LA `).
2. **Recalcula** Ci/Ca e iWUE sustituyendo Γ\* = 40 ppm (constante) por Γ\*(T) según Bernacchi et al. (2001), manteniendo la estructura de la ecuación de Bauters et al. (2020) y **conservando las columnas originales**.
3. **Verifica** que el cálculo reproduce exactamente los valores originales cuando se fuerza Γ\* = 40 ppm, antes de aplicar la corrección.

### Justificación

Bauters et al. (2020) usan Γ\* ≈ 40 ppmv como constante. Esa aproximación es adecuada en Yangambi, que tiene una temperatura estable de 24.5 °C todo el año (Γ\* real ≈ 41.7 ppm, error del 4 %). En el gradiente del NUMEX, con 10 °C de rango térmico, el mismo valor fijo sobreestima Γ\* entre un 25 % (1000 m) y un 117 % (3000 m).

**Nota sobre presión:** Γ\* es fundamentalmente una presión parcial y escala con la presión atmosférica, porque Γ\* = 0.5·O/S_c/o. Como la fracción molar de O₂ (20.95 %) es invariante con la altitud, Γ\* **expresada como fracción molar (ppm) es independiente de la presión**, que es la unidad que requiere la ecuación de discriminación. La variación entre sitios es por tanto enteramente térmica.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 80)

IN_FILE  = Path('data_for_R.xlsx')
OUT_FILE = Path('data_for_R_Bernacchi.xlsx')

df_raw = pd.read_excel(IN_FILE)
print(f'Filas: {df_raw.shape[0]}   Columnas: {df_raw.shape[1]}')

## 1. Limpieza de espacios en blanco

Se detectan y corrigen tres tipos de problema:

- **Nombres de columna** con espacios al inicio/final (`'LA '`).
- **Valores de texto** con espacios al inicio/final (`'Weinmania '`, `'Hedyosmun '`, `'Clarisia '`, `'CaNu '`).
- **Espacios internos dobles** (`'Weinmania  loxensis'` → `'Weinmania loxensis'`), que rompen los agrupamientos igual que los espacios finales pero son invisibles al inspeccionar.

In [ ]:
# --- Diagnostico ANTES de limpiar ---
print('Columnas con espacios sobrantes en el nombre:')
print(' ', [repr(col) for col in df_raw.columns if isinstance(col, str) and col != col.strip()])

def text_cols(frame):
    """Columnas que contienen cadenas de texto."""
    return [col for col in frame.columns
            if frame[col].map(lambda v: isinstance(v, str)).any()]

print('\nValores de texto con espacios sobrantes o dobles:')
for col in text_cols(df_raw):
    vals = df_raw[col].dropna().unique()
    bad = [v for v in vals
           if isinstance(v, str) and (v != v.strip() or '  ' in v)]
    if bad:
        print(f'  {col:10s} -> {[repr(v) for v in bad]}')

In [ ]:
df = df_raw.copy()

# 1a. Nombres de columna
df.columns = [col.strip() if isinstance(col, str) else col for col in df.columns]

# 1b. Valores de texto: quitar espacios externos y colapsar internos multiples
def clean_text(s):
    if not isinstance(s, str):
        return s
    return ' '.join(s.split())   # strip + colapsa cualquier bloque de espacios a uno solo

for col in text_cols(df):
    df[col] = df[col].map(clean_text)

# --- Verificacion DESPUES ---
n_bad = sum(
    1
    for col in text_cols(df)
    for v in df[col].dropna().unique()
    if isinstance(v, str) and (v != v.strip() or '  ' in v)
)
print(f'Valores de texto problematicos tras la limpieza: {n_bad}')
print(f'Columnas con nombre problematico tras la limpieza: '
      f'{sum(1 for col in df.columns if isinstance(col, str) and col != col.strip())}')

print('\nValores unicos tras la limpieza:')
for col in ['name', 'genus', 'specie', 'Site', 'Sub']:
    print(f'  {col:8s}: {sorted(df[col].dropna().unique().tolist())}')

### 1c. Ortografía de los géneros (opcional)

Los géneros aparecen como `Weinmania` y `Hedyosmun`; la grafía aceptada es **Weinmannia** (Cunoniaceae) y **Hedyosmum** (Chloranthaceae). Se corrigen en columnas nuevas (`genus_std`, `name_std`) y se **conservan las originales**, de modo que el cambio es reversible y auditable.

Poner `FIX_SPELLING = False` para desactivarlo.

In [ ]:
FIX_SPELLING = True

SPELLING = {
    'Weinmania': 'Weinmannia',
    'Hedyosmun': 'Hedyosmum',
}

df['genus_std'] = df['genus']
df['name_std']  = df['name']

if FIX_SPELLING:
    for wrong, right in SPELLING.items():
        df['genus_std'] = df['genus_std'].str.replace(rf'\b{wrong}\b', right, regex=True)
        df['name_std']  = df['name_std'].str.replace(rf'\b{wrong}\b', right, regex=True)

# 'specie' guarda a veces el epiteto y a veces el binomio completo -> etiqueta unica y consistente
df['species_label'] = df['name_std']

print(df[['name', 'name_std', 'genus', 'genus_std']].drop_duplicates()
        .sort_values('name_std').to_string(index=False))

## 2. Temperatura media anual por sitio

La columna `mean_annual_temperature_C` del archivo contiene valores **redondeados a entero** (10, 15, 19 °C). Los valores publicados para estos sitios (Tabla 1; Homeier et al. 2013, Wittich et al. 2014) son 19.4, 15.7 y 9.4 °C.

Como Γ\* es sensible a la temperatura, se usa el valor publicado. La columna original se conserva.

> Si más adelante se dispone de temperatura diurna de estación para la temporada de crecimiento, basta con sustituir el diccionario `MAT_SITE`. Ver el análisis de sensibilidad en la sección 6.

In [ ]:
MAT_SITE = {1000: 19.4, 2000: 15.7, 3000: 9.4}   # °C, Tabla 1 del manuscrito

df['MAT_site_C'] = df['elevation_m'].map(MAT_SITE)

chk = (df.groupby('elevation_m')
         .agg(n=('MAT_site_C', 'size'),
              T_archivo=('mean_annual_temperature_C', 'mean'),
              T_usada=('MAT_site_C', 'mean')))
print(chk.to_string())
assert df['MAT_site_C'].notna().all(), 'Hay elevaciones sin temperatura asignada'

## 3. Parámetros del modelo de discriminación

Estructura de la ecuación, idéntica a Bauters et al. (2020, ec. 2–4):

$$\Delta^{13}C_{cell} = a + (b - a)\frac{C_i}{C_a} - \frac{f\,\Gamma^*}{C_a}$$

$$\frac{C_i}{C_a} = \frac{\Delta^{13}C_{cell} - a + f\,\Gamma^*/C_a}{b - a}
\qquad
iWUE = \frac{A}{g_s} = \frac{C_a}{1.6}\left(1 - \frac{C_i}{C_a}\right)$$

con $a = 4.4$ ‰ (O'Leary 1981), $b = 27$ ‰ (Farquhar & Richards 1984) y $f = 12$ ‰ (Farquhar et al. 1982).

Γ\* según Bernacchi et al. (2001), con dependencia de Arrhenius:

$$\Gamma^*(T) = \Gamma^*_{25}\,\exp\!\left[\frac{\Delta H_a\,(T_K - 298.15)}{298.15\; R\; T_K}\right]$$

In [ ]:
# --- Constantes ---
A_FRAC   = 4.4      # ‰, difusion en aire (O'Leary 1981)
B_FRAC   = 27.0     # ‰, carboxilacion efectiva (Farquhar & Richards 1984)
F_FRAC   = 12.0     # ‰, fotorrespiracion (Farquhar et al. 1982)
GSTAR_FIXED = 40.0  # ppm, valor constante usado en Bauters et al. (2020)

R_GAS    = 8.3145   # J mol-1 K-1
GSTAR_25 = 42.75    # umol mol-1 a 25 °C (Bernacchi et al. 2001)
HA_GSTAR = 37830.0  # J mol-1, energia de activacion


def gammastar_bernacchi(tc, gstar25=GSTAR_25, ha=HA_GSTAR):
    """Gamma* (umol mol-1) en funcion de la temperatura en °C.

    Devuelto como fraccion molar, que es independiente de la presion
    atmosferica porque la fraccion molar de O2 es invariante con la altitud.
    """
    tk = np.asarray(tc, dtype=float) + 273.15
    return gstar25 * np.exp(ha * (tk - 298.15) / (298.15 * R_GAS * tk))


def cica_from_delta(delta13c, cair, gstar, a=A_FRAC, b=B_FRAC, f=F_FRAC):
    """Ci/Ca a partir de Delta13C (Bauters et al. 2020, ec. 3)."""
    return (delta13c - a + f * gstar / cair) / (b - a)


def iwue_from_cica(cica, cair):
    """iWUE = A/gs en umol mol-1 (Bauters et al. 2020, ec. 4)."""
    return cair * (1.0 - cica) / 1.6


# Curva de referencia
for t in [24.5, 19.4, 15.7, 9.4]:
    g = gammastar_bernacchi(t)
    print(f'T = {t:5.1f} °C  ->  Gamma* = {g:5.1f} ppm   '
          f'(el valor fijo de 40 ppm se desvia {100*(40-g)/g:+6.1f} %)')

## 4. Verificación: reproducir el cálculo original

Antes de aplicar cualquier corrección se comprueba que las funciones reproducen **exactamente** las columnas `Ci/Ca` e `iWUE (μmol/mol)` del archivo cuando se fuerza Γ\* = 40 ppm. Si esto no cuadra, cualquier comparación posterior carece de sentido.

In [ ]:
cica_check = cica_from_delta(df['Δ13C_cell'], df['Cair'], GSTAR_FIXED)
iwue_check = iwue_from_cica(cica_check, df['Cair'])

d_cica = (cica_check - df['Ci/Ca']).abs().max()
d_iwue = (iwue_check - df['iWUE (μmol/mol)']).abs().max()

print(f'Diferencia maxima en Ci/Ca : {d_cica:.3e}')
print(f'Diferencia maxima en iWUE  : {d_iwue:.3e} umol mol-1')

assert d_cica < 1e-6, 'La reconstruccion de Ci/Ca NO coincide con el archivo'
assert d_iwue < 1e-6, 'La reconstruccion de iWUE NO coincide con el archivo'
print('\nOK: el calculo replica exactamente los valores originales.')

## 5. Recálculo con Γ\*(T)

Se añaden cuatro columnas nuevas y **no se modifica ninguna existente**:

| Columna nueva | Contenido |
|---|---|
| `Gammastar_ppm` | Γ\*(T) según Bernacchi et al. (2001) |
| `CiCa_Bernacchi` | Ci/Ca recalculado |
| `iWUE_Bernacchi (μmol/mol)` | iWUE recalculada |
| `delta_iWUE` | `iWUE_Bernacchi` − `iWUE (μmol/mol)` |

In [ ]:
df['Gammastar_ppm'] = gammastar_bernacchi(df['MAT_site_C'])
df['CiCa_Bernacchi']  = cica_from_delta(df['Δ13C_cell'], df['Cair'], df['Gammastar_ppm'])
df['iWUE_Bernacchi (μmol/mol)'] = iwue_from_cica(df['CiCa_Bernacchi'], df['Cair'])
df['delta_iWUE'] = df['iWUE_Bernacchi (μmol/mol)'] - df['iWUE (μmol/mol)']

resumen = (df.groupby(['elevation_m', 'species_label'])
             .agg(n=('iWUE (μmol/mol)', 'size'),
                  T_C=('MAT_site_C', 'first'),
                  Gammastar=('Gammastar_ppm', 'first'),
                  CiCa_orig=('Ci/Ca', 'mean'),
                  CiCa_bern=('CiCa_Bernacchi', 'mean'),
                  iWUE_orig=('iWUE (μmol/mol)', 'mean'),
                  iWUE_bern=('iWUE_Bernacchi (μmol/mol)', 'mean'),
                  delta=('delta_iWUE', 'mean'))
             .round(3))
print(resumen.to_string())

In [ ]:
# Gradiente altitudinal antes y despues
grad = (df.groupby('elevation_m')[['iWUE (μmol/mol)', 'iWUE_Bernacchi (μmol/mol)']]
          .mean().round(2))
grad.columns = ['iWUE_original', 'iWUE_Bernacchi']
print(grad.to_string())
print(f"\nRango del gradiente  antes: {grad['iWUE_original'].max() - grad['iWUE_original'].min():.1f}"
      f"   despues: {grad['iWUE_Bernacchi'].max() - grad['iWUE_Bernacchi'].min():.1f} umol mol-1")

# Valores fisiologicamente implausibles
for lab, col in [('original ', 'Ci/Ca'), ('Bernacchi', 'CiCa_Bernacchi')]:
    print(f'{lab}: Ci/Ca max = {df[col].max():.3f} | '
          f'n > 0.90: {(df[col] > 0.90).sum():2d} | n > 0.95: {(df[col] > 0.95).sum():2d}')

**Nota importante para los análisis posteriores.** Γ\* se evalúa a nivel de sitio, así que dentro de cada especie la corrección es una **constante aditiva** idéntica para todos los tratamientos. Los coeficientes y p-valores de cualquier modelo `iWUE ~ N + P + (1|Bloque)` ajustado dentro de especie son por tanto **idénticos** antes y después: la constante se absorbe entera en el intercepto. La corrección solo afecta a la comparación de valores absolutos **entre elevaciones**.

(Esto dejaría de ser cierto si se usara temperatura foliar medida por árbol o por parcela.)

## 6. Análisis de sensibilidad

Tres fuentes de incertidumbre paramétrica, propagadas al contraste que importa: el rango del gradiente altitudinal.

In [ ]:
def gradiente(b=B_FRAC, f=F_FRAC, t_offset=0.0):
    gstar = gammastar_bernacchi(df['MAT_site_C'] + t_offset)
    cica  = cica_from_delta(df['Δ13C_cell'], df['Cair'], gstar, b=b, f=f)
    w     = iwue_from_cica(cica, df['Cair'])
    m     = w.groupby(df['elevation_m']).mean()
    return [round(m[z], 1) for z in (1000, 2000, 3000)] + [round(m.max() - m.min(), 1)]

cols = ['1000 m', '2000 m', '3000 m', 'rango']

sens_f = pd.DataFrame([gradiente(f=v) for v in (8, 12, 16)], columns=cols,
                      index=[f'f = {v} ‰' + (' (usado)' if v == 12 else '') for v in (8, 12, 16)])
sens_b = pd.DataFrame([gradiente(b=v) for v in (25, 27, 29)], columns=cols,
                      index=[f'b = {v} ‰' + (' (usado)' if v == 27 else '') for v in (25, 27, 29)])
sens_T = pd.DataFrame([gradiente(t_offset=v) for v in (0, 2, 4, 6)], columns=cols,
                      index=[f'MAT + {v} °C' + (' (usado)' if v == 0 else '') for v in (0, 2, 4, 6)])

print('--- Sensibilidad a f (fotorrespiracion) ---');    print(sens_f.to_string())
print('\n--- Sensibilidad a b (carboxilacion) ---');     print(sens_b.to_string())
print('\n--- Sensibilidad a la temperatura usada ---');  print(sens_T.to_string())

# Variante sin termino fotorrespiratorio (consejo de T. Sibret)
sin_f = gradiente(f=0)
print(f'\nVariante b = 27 ‰ SIN termino f: '
      f'1000 m = {sin_f[0]}, 2000 m = {sin_f[1]}, 3000 m = {sin_f[2]}, rango = {sin_f[3]}')

## 7. Guardar

Se escriben tres hojas: los datos completos, un diccionario de las columnas nuevas, y la tabla de sensibilidad.

In [ ]:
meta = pd.DataFrame([
    ('Gammastar_ppm', 'umol mol-1',
     'Gamma*(T) segun Bernacchi et al. (2001); Gamma*_25 = 42.75, dHa = 37830 J mol-1'),
    ('CiCa_Bernacchi', 'adimensional',
     'Ci/Ca recalculado con Gamma*(T); a = 4.4, b = 27, f = 12 permil'),
    ('iWUE_Bernacchi (μmol/mol)', 'umol mol-1',
     'iWUE = Ca (1 - Ci/Ca) / 1.6 usando CiCa_Bernacchi'),
    ('delta_iWUE', 'umol mol-1',
     'iWUE_Bernacchi menos iWUE original'),
    ('MAT_site_C', '°C',
     'Temperatura media anual publicada por sitio (19.4 / 15.7 / 9.4)'),
    ('genus_std', 'texto',
     'Genero con ortografia corregida (Weinmannia, Hedyosmum)'),
    ('name_std', 'texto',
     'Binomio con ortografia corregida'),
    ('species_label', 'texto',
     'Etiqueta de especie consistente, derivada de name_std'),
], columns=['columna', 'unidad', 'descripcion'])

sens_all = pd.concat([sens_f, sens_b, sens_T])
sens_all.index.name = 'escenario'

with pd.ExcelWriter(OUT_FILE, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='data', index=False)
    meta.to_excel(writer, sheet_name='columnas_nuevas', index=False)
    sens_all.reset_index().to_excel(writer, sheet_name='sensibilidad', index=False)

print(f'Guardado: {OUT_FILE.resolve()}')
print(f'Filas: {df.shape[0]}   Columnas: {df.shape[1]} '
      f'(originales: {df_raw.shape[1]}, nuevas: {df.shape[1] - df_raw.shape[1]})')

In [ ]:
# Comprobacion final de integridad
back = pd.read_excel(OUT_FILE, sheet_name='data')
assert back.shape == df.shape
assert back['iWUE (μmol/mol)'].round(6).equals(df_raw['iWUE (μmol/mol)'].round(6)), \
    'La columna original de iWUE fue alterada'
assert back['iWUE_Bernacchi (μmol/mol)'].notna().all()
print('Verificacion OK: columnas originales intactas y columnas nuevas completas.')
back[['species_label', 'Sub', 'elevation_m', 'MAT_site_C', 'Gammastar_ppm',
      'Ci/Ca', 'CiCa_Bernacchi', 'iWUE (μmol/mol)',
      'iWUE_Bernacchi (μmol/mol)', 'delta_iWUE']].head(8).round(3)

## Referencias

- Bauters, M. et al. (2020). Century-long apparent decrease in intrinsic water-use efficiency with no evidence of progressive nutrient limitation in African tropical forests. *Global Change Biology* 26: 4449–4461. doi:10.1111/gcb.15145
- Bernacchi, C.J., Singsaas, E.L., Pimentel, C., Portis, A.R. & Long, S.P. (2001). Improved temperature response functions for models of Rubisco-limited photosynthesis. *Plant, Cell & Environment* 24: 253–259. doi:10.1111/j.1365-3040.2001.00668.x
- Farquhar, G.D., O'Leary, M.H. & Berry, J.A. (1982). On the relationship between carbon isotope discrimination and the intercellular carbon dioxide concentration in leaves. *Australian Journal of Plant Physiology* 9: 121–137.
- Farquhar, G.D. & Richards, R.A. (1984). Isotopic composition of plant carbon correlates with water-use efficiency of wheat genotypes. *Australian Journal of Plant Physiology* 11: 539–552.
- Lavergne, A. et al. (2020). Historical changes in the stomatal limitation of photosynthesis: empirical support for an optimality principle. *New Phytologist* 225: 2484–2497. doi:10.1111/nph.16314
- O'Leary, M.H. (1981). Carbon isotope fractionation in plants. *Phytochemistry* 20: 553–567.
- Ubierna, N. & Farquhar, G.D. (2014). Advances in measurements and models of photosynthetic carbon isotope discrimination in C3 plants. *Plant, Cell & Environment* 37: 1494–1498.